In [8]:
import numpy as np
import pandas as pd

## Nelson Rule 위배 감지 함수 정의
### __init__
- mean = 타겟 평균
- std = 타겟 표준편차
- ucl = 타겟 평균에서 +3 타겟 표준편차된 관리 한계선
- lcl = 타겟 평균에서 -3 타겟 표준편차된 관리 한계선

### Rule 1 
1개 포인트가 관리한계선 이탈
데이터 관찰 중 UCL보다 크거나  LCL보다 작은 경우 1 return

### Rule 2
9개 연속된 포인트가 한쪽에 치우친 경우 (Target mean과 LCL or Target mean과 UCL 사이)

### Rule 3
6개 연속된 포인트가 지속 증가 or 감소

### Rule 4
14개 연속된 포인트가 위 아래로 교대 진동

### Rule 5
3개 연속된 포인트 중 2개 포인트가 2 sigma 만큼 벗어난 곳에 분포

### Rule 6
5개 연속된 포인트 중 4개 포인트가 1 sigma 만큼 벗어난 곳에 분포

### Rule 7
연속된 15개 포인트가 +/- 1 sigma 영역 내에 촘촘히 뭉친 형태 분포

### Rule 8
연속된 8개 포인트 가 +/- 1 sigma 영역 밖에서만 나타남 (오락가락하는 상황)


In [26]:
class NelsonRules:
    def __init__(self, target_mean, target_std):
        self.mean = target_mean
        self.std = target_std
        self.ucl = target_mean + 3 * target_std
        self.lcl = target_mean - 3 * target_std

    def detect_rule1(self, series):
        '''Rule 1 : 1 point beyond 3-sigma limits.'''
        return (series > self.ucl) | (series < self.lcl)

    def detect_rule2(self, series):
        '''Rule 2: 9 consecutive points on one side of center line.'''
        signals = np.zeros(len(series), dtype=bool) # 규칙 위반 여부 저장 배열 (기본 False)
        above = (series > self.mean).values # 평균보다 큰 값 배열화 
        below = (series < self.mean).values # 평균보다 작은 값 배열화

        for i in range(8, len(series)): #앞에 최소 8개 있어야 현재 데이터 포함 9개 관찰 가능
            if all(above[i-8:i+1]) or all(below[i-8:i+1]): # 9개 전부 평균 위 or 아래인 경우
                signals[i] = True
        return pd.Series(signals, index=series.index)

    def detect_rule3(self, series):
        '''Rule 3: 6 consecutive points steadily increasing or decreasing.'''
        signals = np.zeros(len(series), dtype=bool)
        val = series.values

        for i in range(5, len(series)): # 앞에 최소 5개 있어야 6개 관찰 가능
            diffs = np.diff(val[i-5:i+1]) # 연속 element간 차이 계산
            if all(diffs > 0) or all(diffs < 0):
                signals[i] = True
        return pd.Series(signals, index=series.index)

    def detect_rule4(self, series):
        '''Rule 4: 14 consecutive points alternating up and down.'''
        signals = np.zeros(len(series), dtype=bool)
        val = series.values

        for i in range(13, len(series)): # 앞에 최소 13개 있어야 14개 관찰 가능
            diffs = np.diff(val[i-13:i+1])
            signs = np.sign(diffs)
            alternating = all(signs[j] * signs[j+1] == -1 for j in range(len(signs)-1))
            # 커지고 작아지는 것 반복 - 연속된 차이의 부호가 다름 (곱 = -1)

            if alternating:
                signals[i] = True
        return pd.Series(signals, index=series.index)

    def detect_rule5(self, series):
        '''Rule 5 : 2 out of 3 consecutive points beyond 2-sigma (same side)'''
        signals = np.zeros(len(series), dtype=bool)
        val = series.values

        upper_2sigma = self.mean + 2*self.std
        lower_2sigma = self.mean - 2*self.std

        beyond_upper = val > upper_2sigma
        beyond_lower = val < lower_2sigma

        for i in range(2, len(series)): #앞에 최소 2개 필요 (3개중 2개)
            window_upper = beyond_upper[i-2:i+1]
            window_lower = beyond_lower[i-2:i+1]

            if window_upper.sum() >= 2 or window_lower.sum() >= 2: # 치우친 데이터 2개 이상
                signals[i] = True

        return pd.Series(signals, index=series.index)
    
    def detect_rule6(self, series):
        '''Rule 6: 4 out of 5 consecutive points beyond 1-sigma (same side).'''
        signals = np.zeros(len(series), dtype=bool)
        val = series.values

        upper_1sigma = self.mean + 1 * self.std
        lower_1sigma = self.mean - 1 * self.std

        beyond_upper = val > upper_1sigma
        beyond_lower = val < lower_1sigma

        for i in range(4, len(series)):  # 최소 4개 있어야 5개 관찰 가능
            window_upper = beyond_upper[i-4:i+1]
            window_lower = beyond_lower[i-4:i+1]

            if window_upper.sum() >= 4 or window_lower.sum() >= 4: # 치우친 데이터 4개 이상
                signals[i] = True

        return pd.Series(signals, index=series.index)
        
    def detect_rule7(self, series):
        '''Rule 7: 15 consecutive points within 1-sigma of center line (both sides).'''
        signals = np.zeros(len(series), dtype=bool)
        val = series.values

        upper_1sigma = self.mean + 1 * self.std
        lower_1sigma = self.mean - 1 * self.std

        within_1sigma = (val > lower_1sigma) & (val < upper_1sigma)

        for i in range(14, len(series)):  # 최소 14개 있어야 15개 관찰 가능
            if all(within_1sigma[i-14:i+1]): # 모든 데이터가 범위 이내
                signals[i] = True 

        return pd.Series(signals, index=series.index)

    def detect_rule8(self, series):
        '''Rule 8: 8 consecutive points beyond 1-sigma on either side (none within 1-sigma).'''
        signals = np.zeros(len(series), dtype=bool)
        val = series.values

        upper_1sigma = self.mean + 1 * self.std
        lower_1sigma = self.mean - 1 * self.std

        beyond_1sigma = (val > upper_1sigma) | (val < lower_1sigma)

        for i in range(7, len(series)):  # 최소 7개 있어야 8개 관찰 가능
            if all(beyond_1sigma[i-7:i+1]):
                signals[i] = True

        return pd.Series(signals, index=series.index)

    

## 웨이퍼 가공 Run 데이터 생성
- 40~46 데이터 drift 증가 상황 가정
- 압력 데이터 생성 (평균 150.0 mTorr에서 표준편차 2인 정규분포로 100개 데이터 생성)

In [27]:
np.random.seed(33)
raw_readings = np.random.normal(150, 2, 100)
raw_readings[40:46] = [150.1, 151.2, 152.3, 153.5, 154.8, 156.1]

df = pd.DataFrame({"Wafer_Run":np.arange(1,101), "Pressure":raw_readings})

### Nelson Rule에 의한 OOC 확인


In [28]:
nelson_rule = NelsonRules(target_mean = 150.0, target_std = 2.0)
df["Rule1_OOC"] = nelson_rule.detect_rule1(df["Pressure"])
df["Rule2_OOC"] = nelson_rule.detect_rule2(df["Pressure"])
df["Rule3_OOC"] = nelson_rule.detect_rule3(df["Pressure"])
df["Rule4_OOC"] = nelson_rule.detect_rule4(df["Pressure"])
df["Rule5_OOC"] = nelson_rule.detect_rule5(df["Pressure"])
df["Rule6_OOC"] = nelson_rule.detect_rule6(df["Pressure"])
df["Rule7_OOC"] = nelson_rule.detect_rule7(df["Pressure"])
df["Rule8_OOC"] = nelson_rule.detect_rule8(df["Pressure"])

In [29]:
print(f"Rule 1 Faults: {df['Rule1_OOC'].sum()}")
print(f"Rule 2 Faults: {df['Rule2_OOC'].sum()}")
print(f"Rule 3 Faults: {df['Rule3_OOC'].sum()}")
print(f"Rule 4 Faults: {df['Rule4_OOC'].sum()}")
print(f"Rule 5 Faults: {df['Rule5_OOC'].sum()}")
print(f"Rule 6 Faults: {df['Rule6_OOC'].sum()}")
print(f"Rule 7 Faults: {df['Rule7_OOC'].sum()}")
print(f"Rule 8 Faults: {df['Rule8_OOC'].sum()}")

Rule 1 Faults: 1
Rule 2 Faults: 0
Rule 3 Faults: 1
Rule 4 Faults: 0
Rule 5 Faults: 2
Rule 6 Faults: 2
Rule 7 Faults: 0
Rule 8 Faults: 0
